In [1]:
!pip install -q transformers accelerate peft trl==0.12.0 bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 7.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00:00:0100:01


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model name for fine-tuning
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model with float16 precision and automatic device mapping
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [5]:
import torch 
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name , torch_dtype = torch.float16 , device_map = "auto")

def ask(question, schema, max_new_tokens=128):
    prompt = f"""Given the database schema:
{schema}

Write a SQL query to answer: {question}"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

schema = "Table: employees (id, name, department, salary)"
question = "What is the average salary of employees in the Engineering department?"

print(ask(question, schema))

from datasets import load_dataset

dataset = load_dataset("xlangai/spider")
print(dataset)
print(dataset["train"][0])

dataset = load_dataset("xlangai/spider")
schema_dataset = load_dataset("richardr1126/spider-schema")
schema_lookup = {row["db_id"]: row for row in schema_dataset["train"]}

def format_schema(db_id):
    row = schema_lookup[db_id]
    formatted = f"Schema:\n{row['Schema (values (type))']}"
    if row["Foreign Keys"]:
        formatted += f"\nForeign Keys:\n{row['Foreign Keys']}"
    return formatted

def build_prompt(question, schema_text):
    return f"""Given the database schema:
{schema_text}

Write a SQL query to answer: {question}

Respond with only the SQL query. No explanation, no markdown formatting."""

def format_example(row):
    schema_text = format_schema(row["db_id"])
    prompt = build_prompt(row["question"], schema_text)
    return {
        "messages": [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": row["query"]}
        ]
    }


formatted_dataset = dataset["train"].map(format_example, remove_columns=dataset["train"].column_names)

# print(" Setup complete.")
# print(formatted_dataset)
# print(formatted_dataset[0])

example = formatted_dataset[0]['messages']
templated = tokenizer.apply_chat_template(example, tokenize=False)
# print(templated)













`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


To find the average salary of employees in the Engineering department, you can use the following SQL query:

```sql
SELECT AVG(salary)
FROM employees
WHERE department = 'Engineering';
```

This query selects the average (`AVG`) of the `salary` column from the `employees` table where the `department` is set to `'Engineering'`.


README.md: 0.00B [00:00, ?B/s]

...

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 1034
    })
})
{'db_id': 'department_management', 'query': 'SELECT count(*) FROM head WHERE age  >  56', 'question': 'How many heads of the departments are older than 56 ?', 'query_toks': ['SELECT', 'count', '(', '*', ')', 'FROM', 'head', 'WHERE', 'age', '>', '56'], 'query_toks_no_value': ['select', 'count', '(', '*', ')', 'from', 'head', 'where', 'age', '>', 'value'], 'question_toks': ['How', 'many', 'heads', 'of', 'the', 'departments', 'are', 'older', 'than', '56', '?']}


README.md: 0.00B [00:00, ?B/s]

spider_schema_rows_v2.json: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

In [6]:
!pip install -U torchao
from trl import SFTTrainer , SFTConfig

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.1 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [7]:
from peft import LoraConfig , get_peft_model

lora_config = LoraConfig( r = 16,
                         lora_alpha = 32,
                         lora_dropout = 0.05,
                         target_modules = ['q_proj' , 'v_proj'] ,
                         bias = 'none',
                         task_type = "CAUSAL_LM"
                       )
model = get_peft_model(model , lora_config)
model.print_trainable_parameters()



Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [9]:
def apply_template(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return {"text": text}

formatted_dataset = formatted_dataset.map(apply_template)
eval_dataset = eval_dataset.map(apply_template)

print(formatted_dataset[0]["text"])

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Given the database schema:
Schema:
department : Department_ID (number) , Name (text) , Creation (text) , Ranking (number) , Budget_in_Billions (number) , Num_Employees (number) | head : head_ID (number) , name (text) , born_state (text) , age (number) | management : department_ID (number) , head_ID (number) , temporary_acting (text)
Foreign Keys:
management : head_ID equals head : head_ID | management : department_ID equals department : Department_ID

Write a SQL query to answer: How many heads of the departments are older than 56 ?

Respond with only the SQL query. No explanation, no markdown formatting.<|im_end|>
<|im_start|>assistant
SELECT count(*) FROM head WHERE age  >  56<|im_end|>



In [10]:
smoke_dataset = formatted_dataset.select(range(300))

eval_dataset = dataset["validation"].select(range(200)).map(
    format_example, remove_columns=dataset["validation"].column_names
)

print(smoke_dataset.column_names)

['messages', 'text']


In [11]:
formatted_dataset = formatted_dataset.map(apply_template)
eval_dataset = eval_dataset.map(apply_template)


Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

In [13]:

training_args = SFTConfig(
    output_dir="./nl2sql-lora-full",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    max_seq_length=1024,
    fp16=True, 
    bf16=False,
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset,
    eval_dataset=eval_dataset,
)

trainer.train()

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
100,0.620454,0.867754
200,0.462210,0.910932
300,0.406443,1.075326
400,0.280394,1.069661
500,0.224684,1.217336
600,0.239822,1.297184
700,0.203196,1.338248
800,0.177127,1.419126
900,0.207531,1.412458


KeyboardInterrupt: 

In [14]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, "./nl2sql-lora-full/checkpoint-100")

print("Loaded checkpoint-100 successfully")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded checkpoint-100 successfully


In [18]:
def ask(question, schema, max_new_tokens=128):
    prompt = f"""Given the database schema:
{schema}

Write a SQL query to answer: {question}

Respond with only the SQL query. No explanation, no markdown formatting."""
    
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

In [19]:
schema1 = "Table: employees (id, name, department, salary)"
question1 = "What is the average salary of employees in the Engineering department?"
print("Q1:", ask(question1, schema1))
print()

schema2 = """Table: employees (id, name, department_id, salary)
Table: departments (id, name, budget)"""
question2 = "List the names of departments where the average employee salary exceeds the department's budget divided by 100."
print("Q2:", ask(question2, schema2))

Q1: SELECT AVG(salary) FROM employees WHERE department = "Engineering"

Q2: SELECT T2.name FROM departments AS T1 JOIN employees AS T2 ON T1.id = T2.department_id GROUP BY T2.department_id HAVING AVG(T1.salary) > CAST(ROUND(SUM(T2.salary)/COUNT(T2.salary), 0) AS REAL) / 100


In [21]:
import re

def normalize_sql(query):
    query = query.strip().lower()
    query = re.sub(r'\s+', ' ', query)
    query = query.rstrip(';')
    return query

def exact_match_eval(model, dataset, num_examples=None):
    n = len(dataset) if num_examples is None else min(num_examples, len(dataset))
    correct = 0
    results = []
    
    for i in range(n):
        row = dataset[i]
        user_msg = row["messages"][0]["content"]
        gold_sql = row["messages"][1]["content"]
        
        messages = [{"role": "user", "content": user_msg}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False)
        pred_sql = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        is_match = normalize_sql(pred_sql) == normalize_sql(gold_sql)
        correct += is_match
        results.append({"question": user_msg, "gold": gold_sql, "pred": pred_sql, "match": is_match})
        
        if (i+1) % 50 == 0:
            print(f"{i+1}/{n} done, running accuracy: {correct/(i+1):.3f}")
    
    accuracy = correct / n
    print(f"\nFinal exact-match accuracy: {accuracy:.3f} ({correct}/{n})")
    return accuracy, results

In [22]:
accuracy, results = exact_match_eval(model, eval_dataset, num_examples=200)  # start with 200 for speed, scale up after

50/200 done, running accuracy: 0.280
100/200 done, running accuracy: 0.220
150/200 done, running accuracy: 0.220
200/200 done, running accuracy: 0.215

Final exact-match accuracy: 0.215 (43/200)


In [24]:
for r in results[:15]:
    if not r["match"]:
        print("Q:", r["question"][-150:])  # just the tail (the actual question)
        print("GOLD:", r["gold"])
        print("PRED:", r["pred"])
        print("---")

Q: country, age for all singers ordered by age from the oldest to the youngest.

Respond with only the SQL query. No explanation, no markdown formatting.
GOLD: SELECT name ,  country ,  age FROM singer ORDER BY age DESC
PRED: SELECT Name ,  Country ,  Age FROM singer ORDER BY Age ASC
---
Q: swer: Show the name and the release year of the song by the youngest singer.

Respond with only the SQL query. No explanation, no markdown formatting.
GOLD: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1
PRED: SELECT T1.song_name ,  T2.Year FROM singer AS T1 JOIN concert AS T2 ON T1.Singer_ID  =  T2.Singer_ID ORDER BY T1.Age DESC LIMIT 1
---
Q: at are the names and release years for all the songs of the youngest singer?

Respond with only the SQL query. No explanation, no markdown formatting.
GOLD: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1
PRED: SELECT song_name ,  year FROM concert WHERE age  =  ( SELECT min(age) FROM concert )
---
Q: ert_ID

Wri